# Fiddler GenAI Dashboard Copy (with Custom Chart Order)

Export charts from a dashboard in one **GEN_AI_APP** project and import them to another,
with support for:
* Cross-instance transfers (different Fiddler instances)
* Cross-application transfers (different Applications on same or different instances)
* Remapping chart `application` references to the target Application
* **Custom chart ordering on the new dashboard**

This notebook is the GenAI counterpart to `copying_a_dashboard_with_custom_order.ipynb`
(which targets **ML_MODEL** projects). GenAI dashboards bind to **Applications**, not Models,
and use chart types `GEN_AI_MONITORING` / `GEN_AI_METRICS`.

## How chart ordering works

After exporting charts from the source dashboard, a dedicated cell auto-populates `CHART_ORDER`
with the chart titles in their **current source layout order**. You can then rearrange that list
before running the import and dashboard-creation cells.

Charts are arranged in a **2-column grid** (left-to-right, top-to-bottom):
```
| position 1  | position 2  |
| position 3  | position 4  |
| position 5  | position 6  |
```

## Prerequisites

* Source Fiddler instance URL and API token
* Target Fiddler instance URL and API token (can be same as source)
* Source and target projects must be **GEN_AI_APP** projects
* Source and target Application UUIDs (from GenAI Apps → Application Details)
* Charts that use `metric_source: custom` require a GenAI custom metric with the **same name** on the target
* Charts that filter by evaluator rules / agents expect matching names on the target Application
* Standalone `fiddler-utils` package installed (see install cell below)

In [ ]:
# Install dependencies
# %pip install -q fiddler-client
# %pip install -e /path/to/fiddler-utils   # or: pip install fiddler-utils

# Prefer the standalone fiddler-utils package (source of truth).
# Optionally fall back to a sibling checkout during local development:
import sys
from pathlib import Path

_local_utils = Path("../../fiddler-utils/src")
if _local_utils.exists():
    sys.path.insert(0, str(_local_utils.resolve()))

In [ ]:
import json

import fiddler as fdl
from fiddler_utils import ConnectionManager, ChartManager
from requests import HTTPError, Response
from fiddler.libs.http_client import RequestClient

print(f"Fiddler client version: {fdl.__version__}")
print("fiddler_utils: Successfully imported")

## Configuration

In [ ]:
# Access Token
TOKEN = ""

In [ ]:
# Source Fiddler Instance
SOURCE_URL = ""  # e.g., 'https://source.fiddler.ai'
SOURCE_TOKEN = TOKEN
SOURCE_PROJECT_NAME = ""  # GEN_AI_APP project name
SOURCE_APPLICATION_ID = ""  # Application UUID

# Target Fiddler Instance (can be same as source)
TARGET_URL = ""  # e.g., 'https://target.fiddler.ai'
TARGET_TOKEN = TOKEN
TARGET_PROJECT_NAME = ""  # GEN_AI_APP project name
TARGET_APPLICATION_ID = ""  # Application UUID (must already exist)

# Title for the new dashboard created on the target application
TARGET_DASHBOARD_TITLE = ""  # e.g., 'GenAI Monitoring Copy'

# Chart Export Options
SOURCE_DASHBOARD_ID = ""  # Dashboard UUID to export charts from
CHART_IDS_TO_EXPORT = []  # Optional. e.g., ['uuid1', 'uuid2'] or [] for all from dashboard

## Setup Connections

In [ ]:
conn_mgr = ConnectionManager(log_level="WARNING")
conn_mgr.add("source", url=SOURCE_URL, token=SOURCE_TOKEN)
conn_mgr.add("target", url=TARGET_URL, token=TARGET_TOKEN)

print("✓ Connection manager configured")

## Fetch Source and Target Applications

Resolves projects by name and Applications by UUID via `GET /v3/applications/{id}`.

In [ ]:
def _make_client(url: str, token: str) -> RequestClient:
    return RequestClient(
        base_url=url,
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        },
    )


def _unwrap_data(response):
    body = response.json() if hasattr(response, "json") else response
    if isinstance(body, dict) and "data" in body:
        return body["data"]
    return body


def get_application(client: RequestClient, application_id: str) -> dict:
    """Fetch an Application by UUID."""
    application_id = str(application_id).strip()
    if not application_id:
        raise ValueError("application_id is required")
    response = client.get(url=f"/v3/applications/{application_id}")
    data = _unwrap_data(response)
    if not isinstance(data, dict) or not data.get("id"):
        raise LookupError(f"Application '{application_id}' not found")
    return data


source_client = _make_client(SOURCE_URL, SOURCE_TOKEN)
target_client = _make_client(TARGET_URL, TARGET_TOKEN)

with conn_mgr.use("source"):
    source_project = fdl.Project.from_name(SOURCE_PROJECT_NAME)
    source_application = get_application(source_client, SOURCE_APPLICATION_ID)
    print(
        f"Source project: {source_project.name} (ID: {source_project.id})"
    )
    print(
        f"Source application: {source_application['name']} "
        f"(ID: {source_application['id']})"
    )

with conn_mgr.use("target"):
    target_project = fdl.Project.from_name(TARGET_PROJECT_NAME)
    target_application = get_application(target_client, TARGET_APPLICATION_ID)
    print(
        f"Target project: {target_project.name} (ID: {target_project.id})"
    )
    print(
        f"Target application: {target_application['name']} "
        f"(ID: {target_application['id']})"
    )

## Export/Import Charts

Transfer GenAI charts (`GEN_AI_MONITORING` / `GEN_AI_METRICS`) using `ChartManager`.

**To use this section:**
1. Set `SOURCE_DASHBOARD_ID` in the configuration cell above (find dashboard ID in Fiddler UI URL)
2. OR set `CHART_IDS_TO_EXPORT` to manually specify chart UUIDs
3. Run the cells below to export and import charts

**Note:** Chart API uses unofficial Fiddler endpoints and may change without notice.

In [ ]:
print("\n" + "=" * 60)
print("INITIALIZING CHART MANAGERS")
print("=" * 60)

source_chart_mgr = ChartManager(url=SOURCE_URL, token=SOURCE_TOKEN)
target_chart_mgr = ChartManager(url=TARGET_URL, token=TARGET_TOKEN)

print("\n✓ Source ChartManager initialized")
print("✓ Target ChartManager initialized")

In [ ]:
print("\n" + "=" * 60)
print("EXPORTING CHARTS")
print("=" * 60)

exported_charts = []

if SOURCE_DASHBOARD_ID:
    print(f"\nExporting charts from dashboard: {SOURCE_DASHBOARD_ID}")
elif CHART_IDS_TO_EXPORT:
    print(f"\nExporting {len(CHART_IDS_TO_EXPORT)} charts by ID")
else:
    print("\n⚠️  No SOURCE_DASHBOARD_ID or CHART_IDS_TO_EXPORT specified.")
    print("   Set one of these in the configuration cell to export charts.")

if SOURCE_DASHBOARD_ID or CHART_IDS_TO_EXPORT:
    with conn_mgr.use("source"):
        try:
            exported_charts = source_chart_mgr.export_charts(
                dashboard_id=SOURCE_DASHBOARD_ID if SOURCE_DASHBOARD_ID else None,
                chart_ids=CHART_IDS_TO_EXPORT if CHART_IDS_TO_EXPORT else None,
            )

            print(f"\n✓ Exported {len(exported_charts)} chart(s)\n")

            for i, chart in enumerate(exported_charts, 1):
                data_source = chart.get("data_source", {})
                query_type = data_source.get("query_type") or chart.get(
                    "query_type", "unknown"
                )
                print(f"{i}. {chart.get('title', 'Untitled')}")
                print(f"   Type: {query_type}")

                for query in data_source.get("queries", []):
                    deps = [
                        f"metric_source={query.get('metric_source')}",
                        f"metric_name={query.get('metric_name')}",
                    ]
                    if query.get("custom_metric_name"):
                        deps.append(
                            f"custom_metric={query['custom_metric_name']}"
                        )
                    filters = query.get("filters") or {}
                    if filters.get("agents"):
                        deps.append(f"agents={filters['agents']}")
                    if filters.get("evaluator_rules"):
                        rule_names = [
                            r.get("name")
                            for r in filters["evaluator_rules"]
                            if isinstance(r, dict)
                        ]
                        deps.append(f"evaluator_rules={rule_names}")
                    metric_params = query.get("metric_params") or {}
                    if metric_params.get("rule_name"):
                        deps.append(f"rule_name={metric_params['rule_name']}")
                    print(f"   Query deps: {', '.join(deps)}")
                print()
        except Exception as e:
            print(f"\n❌ Failed to export charts: {e}")
            exported_charts = []

## Set Dashboard Chart Order

Charts from the source dashboard are printed with a number when exported above.
Use those numbers in `CHART_ORDER` to control which charts appear on the new dashboard and in what order.

**To set the order:**
1. Run the first cell below — it prints all available charts with their position numbers.
2. Edit `CHART_ORDER` in the second cell, using those numbers in your desired sequence.
3. Run the second cell to see a grid preview of the resulting layout — the numbered reference stays visible above it.
4. Repeat steps 2–3 until satisfied, then continue to the import cells.

**Rules:**
- Only charts whose number appears in `CHART_ORDER` will be included — others are excluded.
- Leave `CHART_ORDER = []` to include all charts in their original source order.
- Duplicate numbers and out-of-range numbers will be flagged.

In [ ]:
# Prints the source dashboard charts with their position numbers.
# Use the numbers shown here when setting CHART_ORDER in the cell below.

exported_titles = [chart.get("title", "Untitled") for chart in exported_charts]
n = len(exported_titles)

if not exported_titles:
    print("⚠️  No charts found. Check that SOURCE_DASHBOARD_ID or CHART_IDS_TO_EXPORT is set.")
else:
    print("=" * 60)
    print("AVAILABLE CHARTS")
    print("=" * 60)
    for i, title in enumerate(exported_titles, 1):
        print(f"  {i:>2}. {title}")
    print()
    print("Set CHART_ORDER in the cell below using these numbers, then re-run that cell")
    print("to see a preview of the resulting dashboard layout.")

In [ ]:
# Step 1: Enter your preferred chart order using the numbers from the cell above.
#         Any charts you don't include here will be appended at the end (see Step 2).
#         Leave empty [] to keep all charts in their original source order.
#
MY_CHART_ORDER = []

# Step 2: Any charts not listed in MY_CHART_ORDER are automatically appended at the end.
#
CHART_ORDER = MY_CHART_ORDER + [i for i in range(1, n + 1) if i not in MY_CHART_ORDER]

# ── Resolve numbers to titles and validate ────────────────────────────────────

if not exported_titles:
    print("⚠️  No charts found. Run the export cell first.")
else:
    if not CHART_ORDER:
        final_chart_titles = exported_titles
    else:
        out_of_range = [x for x in CHART_ORDER if not (1 <= x <= n)]
        if out_of_range:
            print(f"⚠️  Out-of-range chart numbers (valid range is 1–{n}): {out_of_range}")

        seen, duplicates = set(), []
        for x in CHART_ORDER:
            if x in seen:
                duplicates.append(x)
            seen.add(x)
        if duplicates:
            print(f"⚠️  Duplicate chart numbers (each number should appear once): {duplicates}")

        final_chart_titles = [
            exported_titles[x - 1] for x in CHART_ORDER if 1 <= x <= n
        ]

    # ── Grid preview ─────────────────────────────────────────────────────────
    num_prefix_len = 5
    col_width = max((len(t) for t in final_chart_titles), default=0) + num_prefix_len + 1
    col_width = max(col_width, 24)

    h_div = "─" * col_width
    top = f"┌{h_div}┬{h_div}┐"
    mid = f"├{h_div}┼{h_div}┤"
    bottom = f"└{h_div}┴{h_div}┘"

    def padded(pos, text):
        if text:
            content = f"{pos}. {text}"
        else:
            content = ""
        return f" {content:<{col_width - 1}}"

    print("\nDashboard layout preview:\n")
    rows = [final_chart_titles[i : i + 2] for i in range(0, len(final_chart_titles), 2)]
    pos = 1
    for r, row in enumerate(rows):
        print(top if r == 0 else mid)
        left = padded(pos, row[0]) if len(row) > 0 else padded("", "")
        right = padded(pos + 1, row[1]) if len(row) > 1 else padded("", "")
        print(f"│{left}│{right}│")
        pos += 2
    print(bottom)
    print(f"\n{len(final_chart_titles)} chart(s) selected.")
    print("\n" + "─" * 60)
    print("If this looks correct, continue to the next cell to import charts.")
    print("To adjust: edit MY_CHART_ORDER above and re-run this cell.")
    print("─" * 60)

In [ ]:
if not exported_charts:
    print("\n⊘ No charts to import. Skipping import.")
    chart_result = {"successful": 0, "failed": 0, "errors": [], "imported_charts": []}
    imported_charts = []
else:
    # Import only charts selected by CHART_ORDER / final_chart_titles
    titles_to_import = set(final_chart_titles)
    charts_to_import = [
        c for c in exported_charts if c.get("title", "Untitled") in titles_to_import
    ]

    print("\n" + "=" * 60)
    print("IMPORTING GENAI CHARTS")
    print("=" * 60)

    with conn_mgr.use("target"):
        chart_result = target_chart_mgr.import_genai_charts(
            target_project_id=target_project.id,
            target_application_id=target_application["id"],
            charts=charts_to_import,
            dry_run=False,
        )

        print("\nResults:")
        print(f"  ✅ Successfully imported: {chart_result['successful']}")
        print(f"  ❌ Failed: {chart_result['failed']}")

        if chart_result.get("errors"):
            print("\n  Errors encountered:")
            for title, error in chart_result["errors"]:
                print(f"    • {title}")
                print(f"      {error}")

        if chart_result["successful"] > 0:
            print(
                f"\n✓ Successfully imported {chart_result['successful']} "
                f"chart(s) to target application"
            )

        imported_charts = chart_result["imported_charts"]

## Package Charts Into a New Default Dashboard

Creates a new dashboard on the target **GEN_AI_APP** project using the chart order defined
in `CHART_ORDER` above. The dashboard is set as the default for the target Application via
`PUT /v3/applications/{id}/default-dashboard`.

In [ ]:
client = RequestClient(
    TARGET_URL,
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {TARGET_TOKEN}",
    },
)


def generate_chart_layout(chart_title, x, y):
    """Build a single layout entry for one chart at grid position (x, y)."""
    return {
        "chart_title": chart_title,
        "grid_props": {
            "height": 1,
            "position_x": x,
            "position_y": y,
            "width": 1,
        },
    }


def generate_chart_layouts(chart_titles):
    """Arrange chart titles into a 2-column grid layout (left-to-right, top-to-bottom)."""
    chart_layouts = []
    for i, chart_title in enumerate(chart_titles):
        x = i % 2
        y = i // 2
        chart_layouts.append(generate_chart_layout(chart_title, x, y))
    return chart_layouts


def create_dashboard(project_name: str, charts: list, dashboard: dict) -> Response:
    dashboards_url = "v2/dashboards"

    dashboard["organization_name"] = fdl.conn.organization_name
    dashboard["project_name"] = project_name

    try:
        chart_title_to_uuid = {chart["title"]: chart["id"] for chart in charts}

        for saved_chart in dashboard.get("layouts", []):
            chart_title = saved_chart.get("chart_title")
            saved_chart["chart_uuid"] = chart_title_to_uuid.get(chart_title)
            saved_chart.pop("chart_title", None)

        dashboard_resp: Response = client.post(url=dashboards_url, data=dashboard)

        dashboard_uuid = dashboard_resp.json()["data"].get("uuid")
        default_dashboard_url = (
            f"v3/applications/{target_application['id']}/default-dashboard"
        )
        client.put(
            url=default_dashboard_url,
            data={"dashboard_uuid": dashboard_uuid},
        )

        return dashboard_resp

    except HTTPError as hex:
        print(
            f"HTTPError occurred: {hex.response.text} "
            f"with error code {hex.response.status_code}"
        )
        raise hex


def update_default_dashboard(chart_titles):
    """Create a new dashboard from the given chart title order and set it as default."""
    if not chart_titles:
        print("⚠️  No chart titles selected. Skipping dashboard creation.")
        return

    if not imported_charts:
        print("⚠️  No imported charts available. Run the import cell first.")
        return

    dashboard = {
        "layouts": generate_chart_layouts(chart_titles),
        "options": {
            "filters": {"time_label": "30d", "time_zone": "UTC"}
        },
        "organization_name": fdl.conn.organization_name,
        "project_name": TARGET_PROJECT_NAME,
        "title": TARGET_DASHBOARD_TITLE,
    }

    with conn_mgr.use("target"):
        dashboard_response = create_dashboard(
            project_name=TARGET_PROJECT_NAME,
            charts=imported_charts,
            dashboard=dashboard,
        )

        dashboard_uuid = dashboard_response.json()["data"]["uuid"]
        print(f"  ✅ Created dashboard: {TARGET_DASHBOARD_TITLE} ({dashboard_uuid})")
        print(
            f"  ✅ Set as default for application '{target_application['name']}'"
        )


# final_chart_titles was resolved in the "Set Dashboard Chart Order" cell above.
update_default_dashboard(final_chart_titles)